In [1]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from pathlib import Path
import sys

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
PROJECT_PATH = Path("/content/drive/MyDrive/UAH_Project")

SRC_PATH = PROJECT_PATH / "src"

PROCESSED_PATH = PROJECT_PATH / "datasets" / "processed"

sys.path.append(str(SRC_PATH))

In [5]:
data = np.load(
    PROCESSED_PATH / "uah_dataset.npz"
)

X = data["X"]
y = data["y"]
groups = data["groups"]

In [6]:
print("=" * 60)
print("Dataset Information")
print("=" * 60)

print("X :", X.shape)
print("y :", y.shape)
print("groups :", groups.shape)

print()

print("Class Distribution")

classes, counts = np.unique(y, return_counts=True)

for c, n in zip(classes, counts):
    print(f"Class {c}: {n}")

Dataset Information
X : (30676, 120, 13)
y : (30676,)
groups : (30676,)

Class Distribution
Class 0: 12991
Class 1: 9846
Class 2: 7839


# ==========================================================
# EXPERIMENT 1
# REMOVE FRONT_DISTANCE & RELATIVE_SPEED
# ==========================================================

In [7]:
feature_names = [
    "acc_x",
    "acc_y",
    "acc_z",
    "roll",
    "pitch",
    "yaw",
    "speed",
    "heading",
    "lane_offset",
    "phi",
    "front_distance",
    "relative_speed",
    "vehicle_state",
]

print(feature_names)

['acc_x', 'acc_y', 'acc_z', 'roll', 'pitch', 'yaw', 'speed', 'heading', 'lane_offset', 'phi', 'front_distance', 'relative_speed', 'vehicle_state']


In [8]:
remove_features = [
    "front_distance",
    "relative_speed",
]

keep_indices = [
    i for i, f in enumerate(feature_names)
    if f not in remove_features
]

print(keep_indices)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12]


In [9]:
X_new = X[:, :, keep_indices]

print(X.shape)
print(X_new.shape)

(30676, 120, 13)
(30676, 120, 11)


# ==========================================================
# 2. TRAIN / TEST SPLIT
# ==========================================================

In [15]:
from sklearn.model_selection import train_test_split
import numpy as np

unique_groups = np.unique(groups)

train_groups, test_groups = train_test_split(
    unique_groups,
    test_size=0.20,
    random_state=42,
)

train_mask = np.isin(groups, train_groups)
test_mask = np.isin(groups, test_groups)

X_train = X_new[train_mask]
y_train = y[train_mask]

X_test = X_new[test_mask]
y_test = y[test_mask]

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (24115, 120, 11)
Test : (6561, 120, 11)


In [16]:
import importlib
import trainer

importlib.reload(trainer)

from trainer import create_dataloader

train_loader = create_dataloader(
    X_train,
    y_train,
    batch_size=32,
    shuffle=True,
)

test_loader = create_dataloader(
    X_test,
    y_test,
    batch_size=32,
    shuffle=False,
)

print("Train Loader:", len(train_loader))
print("Test Loader :", len(test_loader))

Train Loader: 754
Test Loader : 206


In [17]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)

print(class_weights)

[0.69680421 1.28531073 1.27088274]


In [18]:
import importlib
import lstm_model
import trainer

importlib.reload(lstm_model)
importlib.reload(trainer)

from lstm_model import LSTMClassifier
from trainer import fit

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model_feature = LSTMClassifier().to(device)

weights = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(
    weight=weights
)

optimizer = torch.optim.Adam(
    model_feature.parameters(),
    lr=1e-4,
)

print(model_feature)

LSTMClassifier(
  (lstm): LSTM(11, 64, num_layers=2, batch_first=True, dropout=0.3)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=3, bias=True)
  )
)


In [22]:
# ==========================================================
# CONFIGURATION
# ==========================================================

BATCH_SIZE = 32
LEARNING_RATE = 1e-4
EPOCHS = 20
PATIENCE = 3

In [23]:
history_feature = fit(
    model=model_feature,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
)

Epoch 1/20 | Train Loss: 0.8390 | Train Acc: 0.5142 | Val Loss: 0.9941 | Val Acc: 0.5162
Epoch 2/20 | Train Loss: 0.8197 | Train Acc: 0.5273 | Val Loss: 0.9190 | Val Acc: 0.6081
Epoch 3/20 | Train Loss: 0.8028 | Train Acc: 0.5334 | Val Loss: 1.0967 | Val Acc: 0.5580
Epoch 4/20 | Train Loss: 0.7987 | Train Acc: 0.5373 | Val Loss: 1.4045 | Val Acc: 0.4816
Epoch 5/20 | Train Loss: 0.7854 | Train Acc: 0.5420 | Val Loss: 1.0427 | Val Acc: 0.5321

Early stopping at epoch 5
Best Validation Accuracy : 0.6081
